# Play tic-tac-toe against a DQN that taught itself

AI 4200, computational processes lecture. One Q-network learns tic-tac-toe by playing **both sides against itself**.
Nobody tells it the rules: the only signal is **+1 when a move makes three in a row**.

1. Run the **setup** and **train** cells (about 3–5 minutes on Colab's free CPU; no GPU needed).
2. Scroll down and **play against it**.

Same method as the slides: target `q = r − γ · max Q̂(s′)`, squared-error loss, replay memory, and a frozen copy for the targets.

## Setup: the game

In [ ]:
import random
from functools import lru_cache

import numpy as np
import torch
import torch.nn as nn

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]


def winner(board):
    """+1 if X has three in a row, -1 if O does, else 0."""
    for i, j, k in LINES:
        if board[i] != 0 and board[i] == board[j] == board[k]:
            return board[i]
    return 0


def legal_moves(board):
    """Indices of the empty squares (0..8, row by row)."""
    return [i for i in range(9) if board[i] == 0]


def play(board, square, player):
    """New board with `player` (+1 for X, -1 for O) on `square`."""
    return board[:square] + (player,) + board[square + 1:]

## The Q-network

The input is the board **from the point of view of the player to move**: +1 mine, −1 theirs, 0 empty.
That is how one network plays both X and O: on O's turn we just flip the signs.
The output is Q̂, one score per square: the predicted result of playing there (+1 win, 0 draw, −1 loss).

In [ ]:
def make_q_network():
    """9 board numbers in, 9 scores (one per square) out."""
    return nn.Sequential(
        nn.Linear(9, 128), nn.SiLU(),
        nn.Linear(128, 128), nn.SiLU(),
        nn.Linear(128, 9),
    )


q_net = make_q_network()
frozen_copy = make_q_network()
frozen_copy.load_state_dict(q_net.state_dict())
optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)


def view(board, player):
    """The board as `player` sees it: +1 mine, -1 theirs, 0 empty."""
    return torch.tensor([v * player for v in board], dtype=torch.float32)


def scores(board, player, model=q_net):
    """Q-hat for every square, from `player`'s point of view."""
    with torch.no_grad():
        return model(view(board, player)).tolist()


def best_move(board, player):
    """The empty square with the highest score."""
    q_hat = scores(board, player)
    return max(legal_moves(board), key=lambda i: q_hat[i])

## Opponents for checking progress

A random player, and a **perfect** player (full minimax search). Against a perfect player the best you can do is a draw.
These are only used to measure the network; it never trains against them.

In [ ]:
@lru_cache(maxsize=None)
def minimax_value(board, player):
    """Value of the position for `player` to move, with perfect play: +1, 0 or -1."""
    w = winner(board)
    if w:
        return w * player
    if 0 not in board:
        return 0
    return max(-minimax_value(play(board, i, player), -player) for i in legal_moves(board))


def perfect_player(board, player):
    """A random choice among the moves that minimax says are best."""
    values = {i: -minimax_value(play(board, i, player), -player) for i in legal_moves(board)}
    best = max(values.values())
    return random.choice([i for i, v in values.items() if v == best])


def random_player(board, player):
    return random.choice(legal_moves(board))


def match(opponent, games=100):
    """The network plays `games` games (half as X, half as O). Returns win, draw, loss fractions."""
    results = []
    for game in range(games):
        network_side = 1 if game % 2 == 0 else -1
        board, player = (0,) * 9, 1
        while True:
            move = best_move(board, player) if player == network_side else opponent(board, player)
            board = play(board, move, player)
            w = winner(board)
            if w or 0 not in board:
                results.append(0 if not w else (1 if w == network_side else -1))
                break
            player = -player
    return results.count(1) / games, results.count(0) / games, results.count(-1) / games

## Train by self-play

Each move is stored as (s, a, r, s′). After every move: one gradient step on 128 moves drawn at random from memory.

- If the move ended the game: `q = r` (+1 for a win, 0 for a draw).
- Otherwise: `q = −γ · max Q̂(s′)`, the opponent's best score on the next board, sign-flipped, from the frozen copy.
- The frozen copy is overwritten with the current weights every 500 updates.

**Start this cell and let it run (3–5 min).** Watch the draw rate against the perfect player climb.

In [ ]:
GAMES = 10_000          # 10,000 is plenty; our lecture run used 30,000
GAMMA = 0.95
BATCH = 128
SYNC_EVERY = 500
MEMORY_SIZE = 50_000
EXPLORE_START, EXPLORE_END, EXPLORE_GAMES = 1.0, 0.1, 5_000

memory = []
updates = 0


def train_step():
    """One gradient step on a random batch of remembered moves."""
    global updates
    batch = random.sample(memory, BATCH)
    states = torch.stack([m[0] for m in batch])
    actions = torch.tensor([m[1] for m in batch])
    rewards = torch.tensor([m[2] for m in batch])
    next_states = torch.stack([m[3] for m in batch])
    done = torch.tensor([m[4] for m in batch], dtype=torch.float32)
    filled = torch.tensor([m[5] for m in batch])           # squares the opponent can't play

    with torch.no_grad():
        next_best = frozen_copy(next_states).masked_fill(filled, -1e9).max(1).values
        next_best = torch.where(done > 0, torch.zeros_like(next_best), next_best)
        target = rewards - GAMMA * next_best                 # q = r - gamma * max Q(s')

    q_sa = q_net(states).gather(1, actions[:, None]).squeeze(1)
    loss = ((q_sa - target) ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    updates += 1
    if updates % SYNC_EVERY == 0:
        frozen_copy.load_state_dict(q_net.state_dict())   # total swap, not an average


for game in range(1, GAMES + 1):
    explore = max(EXPLORE_END, EXPLORE_START - (EXPLORE_START - EXPLORE_END) * game / EXPLORE_GAMES)
    board, player = (0,) * 9, 1
    while True:
        move = random.choice(legal_moves(board)) if random.random() < explore else best_move(board, player)
        next_board = play(board, move, player)
        won = winner(next_board) == player
        done = won or 0 not in next_board
        reward = 1.0 if won else 0.0
        filled = [i not in legal_moves(next_board) for i in range(9)]
        memory.append((view(board, player), move, reward, view(next_board, -player), done, filled))
        if len(memory) > MEMORY_SIZE:
            memory.pop(0)
        if len(memory) >= BATCH:
            train_step()
        if done:
            break
        board, player = next_board, -player

    if game % 1000 == 0:
        w, d, l = match(random_player)
        pw, pd, pl = match(perfect_player)
        print(f"game {game:>6,}   vs random: win {w:.0%}  loss {l:.0%}    vs perfect: draw {pd:.0%}  loss {pl:.0%}")

print("Done. Scroll down and play it.")

## Play against it

Pick a side and click a square. Tick **show its scores** to see Q̂ for each empty square from the network's point of view
before it moves: it plays the highest one.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

state = {"board": (0,) * 9, "human": 1, "over": False}
buttons = [widgets.Button(layout=widgets.Layout(width="72px", height="72px")) for _ in range(9)]
status = widgets.HTML()
side = widgets.ToggleButtons(options=[("I play X (first)", 1), ("I play O", -1)], value=1)
show_scores = widgets.Checkbox(value=False, description="show its scores")
new_game = widgets.Button(description="New game", button_style="primary")


def redraw():
    board = state["board"]
    q_hat = scores(board, -state["human"]) if show_scores.value and not state["over"] else None
    for i, button in enumerate(buttons):
        mark = {1: "X", -1: "O", 0: ""}[board[i]]
        if not mark and q_hat is not None:
            mark = f"{q_hat[i]:+.2f}"
        button.description = mark
        button.disabled = state["over"] or board[i] != 0
        button.style.button_color = {1: "#DCE5FA", -1: "#FBE3D2", 0: None}[board[i]]
        button.style.font_weight = "bold"


def finish_if_over():
    board = state["board"]
    w = winner(board)
    if w or 0 not in board:
        state["over"] = True
        if w == state["human"]:
            status.value = "<b>You win!</b> (It still has some learning to do.)"
        elif w:
            status.value = "<b>It wins.</b>"
        else:
            status.value = "<b>Draw.</b> Against perfect play, that is the best anyone can do."
        return True
    return False


def network_moves():
    machine = -state["human"]
    state["board"] = play(state["board"], best_move(state["board"], machine), machine)
    if not finish_if_over():
        status.value = "Your move."


def on_click(button):
    square = buttons.index(button)
    if state["over"] or state["board"][square] != 0:
        return
    state["board"] = play(state["board"], square, state["human"])
    if not finish_if_over():
        network_moves()
    redraw()


def start(_=None):
    state.update(board=(0,) * 9, human=side.value, over=False)
    status.value = "Your move."
    if state["human"] == -1:
        network_moves()
    redraw()


for button in buttons:
    button.on_click(on_click)
new_game.on_click(start)
show_scores.observe(lambda change: redraw(), names="value")

start()
display(widgets.VBox([
    widgets.HBox([side, new_game]),
    show_scores,
    widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat(3, 76px)")),
    status,
]))